# 07 — O GPT nativo do Lab-IA

Os cadernos 01–06 usam HuggingFace Transformers + PEFT. O núcleo do Lab-IA tem uma
implementação **própria** de GPT (`labia.models.gpt`) em que a arquitetura é objeto de
estudo: dá para trocar normalização (LayerNorm ↔ RMSNorm), posição (aprendido ↔ RoPE),
MLP (GELU ↔ SwiGLU), atenção (MHA ↔ GQA) e FFN (densa ↔ MoE) por **uma variável da config**.

Pré-requisito: caderno 01 executado (usa `data/corpus.txt` e `artifacts/tokenizer_puro`).


In [ ]:
from pathlib import Path
import torch
from tokenizers import Tokenizer

from labia.models.gpt import ConfigGPT, GPT, gerar as gerar_com_prompt
from labia.trainer.dados import dividir_corpus, montar_dataset

ARQ_TOKENIZER = Path('artifacts/tokenizer_puro/tokenizer.json')
if not ARQ_TOKENIZER.is_file():
    raise FileNotFoundError('Tokenizer ausente. Execute primeiro o caderno 01 (ele salva artifacts/tokenizer_puro).')
tok = Tokenizer.from_file(str(ARQ_TOKENIZER))
corpus = Path('data/corpus.txt').read_text(encoding='utf-8')
print('vocab:', tok.get_vocab_size(), '| corpus com', len(corpus), 'caracteres')


## 1. Arquitetura como configuração

`ConfigGPT` expõe exatamente as escolhas que separaram GPT-2 (2019) dos modelos atuais.
Rode primeiro com RMSNorm + RoPE + SwiGLU; nos exercícios você volta para as opções do
GPT-2 original e compara a mesma métrica.

In [ ]:
JANELA = 128
torch.manual_seed(42)
config = ConfigGPT(
    vocab=tok.get_vocab_size(), dim=128, camadas=4, cabecas=4,
    janela_ctx=JANELA, abandono=0.1,
    norm='rmsnorm',   # 'layernorm' = GPT-2 original
    pos='rope',       # 'aprendido'  = GPT-2 original
    mlp='swiglu',     # 'gelu'         = FFN original
)
modelo = GPT(config)
modelo.init_pesos(semente=42)
disp = 'cuda' if torch.cuda.is_available() else 'cpu'
modelo = modelo.to(disp)
print(f'{modelo.contar_parametros()/1e6:.2f} M parâmetros | dispositivo: {disp}')


## 2. Dataset pelo próprio núcleo

`dividir_corpus` separa trem/validação por parágrafos com semente fixa e
`montar_dataset` produz os pares *(entrada → próximo token)* — o deslocamento
que o `Trainer` do caderno 01 escondia acontece aqui explicitamente.

In [ ]:
trem_texto, val_texto = dividir_corpus(corpus, semente=42)
x_trem, y_trem = montar_dataset(tok, trem_texto, JANELA, stride=JANELA // 2)
x_val, y_val = montar_dataset(tok, val_texto, JANELA)
print(f'trem: {tuple(x_trem.shape)} | validação: {tuple(x_val.shape)}')

@torch.no_grad()
def perda_media(x, y, lote=32, max_lotes=16):
    modelo.eval()
    perdas = []
    for i in range(min(max_lotes, -(-len(x) // lote))):
        _, p = modelo(x[i * lote:(i + 1) * lote].to(disp), y[i * lote:(i + 1) * lote].to(disp))
        perdas.append(float(p))
    return sum(perdas) / len(perdas)

print(f'perda inicial (modelo aleatório): {perda_media(x_val, y_val):.3f}')


## 3. Treino na mão

240 passos de AdamW com clip de gradiente — o mínimo que o `Trainer` faz por trás.
Repare como a perda de validação despenca: o corpus é repetitivo, e isso é proposital
para o smoke test (não confunda decorar variações de 6 frases com linguagem real).

In [ ]:
otim = torch.optim.AdamW(modelo.parameters(), lr=3e-4, betas=(0.9, 0.98), weight_decay=0.1)
L, PASSOS = 32, 240
modelo.train()
for passo in range(PASSOS):
    ini = (passo * L) % max(1, len(x_trem) - L)
    _, perda = modelo(x_trem[ini:ini + L].to(disp), y_trem[ini:ini + L].to(disp))
    otim.zero_grad(set_to_none=True)
    perda.backward()
    torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
    otim.step()
    if (passo + 1) % 60 == 0:
        val = perda_media(x_val, y_val)
        print(f'passo {passo + 1:3d} | perda trem {float(perda):.3f} | val {val:.3f}')
        modelo.train()


## 4. Geração com o `gerar()` do núcleo

`labia.models.gpt.gerar` aplica temperatura + top-k e devolve **só a continuação**.

In [ ]:
modelo.eval()
saida = gerar_com_prompt(modelo, 'Modelos de linguagem', tok, passos_max=60, temperatura=0.8, topo_k=40, semente=7)
print(saida)


## Exercícios

1. Troque `norm`, `pos` e `mlp` para os valores do GPT-2 original e registre as duas
   perdas de validação no mesmo caderno — o que mudou?
2. Reduza `cabecas` para 2 e aumente `n_cabecas_kv` para 1 (GQA): mesmo custo, menos cache.
3. Gere com `temperatura=0` (determinístico) e compare com a amostragem.
4. Continue para o caderno 09, onde a FFN vira MoE.